In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: []


2026-08-16 13:38:34.891676: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:3]:
            print(f"{indent}  {f}")

input/
  datasets/
    shivamagarwal29/
      cow-lumpy-disease-dataset/
        lumpycows/
        healthycows/


In [4]:
DATASET_PATH = "/kaggle/input/datasets/shivamagarwal29/cow-lumpy-disease-dataset"

print("Path exists:", os.path.exists(DATASET_PATH))
print("Classes:", os.listdir(DATASET_PATH))

Path exists: True
Classes: ['lumpycows', 'healthycows']


In [5]:
def count_images(folder):
    count = 0
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                count += 1
    return count

lumpy_dir = os.path.join(DATASET_PATH, "lumpycows")
healthy_dir = os.path.join(DATASET_PATH, "healthycows")

print("Lumpy cow images:", count_images(lumpy_dir))
print("Healthy cow images:", count_images(healthy_dir))

Lumpy cow images: 421
Healthy cow images: 515


In [7]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds_animal = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    color_mode="rgb"
)

val_ds_animal = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    color_mode="rgb"
)

class_names_animal = train_ds_animal.class_names
NUM_CLASSES_ANIMAL = len(class_names_animal)
print("Classes:", class_names_animal)
print("Number of classes:", NUM_CLASSES_ANIMAL)

Found 936 files belonging to 2 classes.
Using 749 files for training.
Found 936 files belonging to 2 classes.
Using 187 files for validation.
Classes: ['healthycows', 'lumpycows']
Number of classes: 2


In [8]:
images, labels = next(iter(train_ds_animal))
print("Image shape:", images.shape)
print("Label shape:", labels.shape)

Image shape: (32, 224, 224, 3)
Label shape: (32, 2)


In [9]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds_animal = train_ds_animal.cache().prefetch(buffer_size=AUTOTUNE)
val_ds_animal = val_ds_animal.cache().prefetch(buffer_size=AUTOTUNE)
print("Dataset optimization complete.")

Dataset optimization complete.


In [10]:
data_augmentation_animal = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation_animal")
print("Data augmentation ready.")

Data augmentation ready.


In [11]:
base_model_animal = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model_animal.trainable = False

inputs_animal = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation_animal(inputs_animal)
x = base_model_animal(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs_animal = tf.keras.layers.Dense(NUM_CLASSES_ANIMAL, activation="softmax")(x)

model_animal = tf.keras.Model(inputs_animal, outputs_animal)
model_animal.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation_animal        │ (None, 224, 224, 3)    │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         2,562 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,052,133 (15.46 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [12]:
model_animal.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
print("Model compiled successfully.")

Model compiled successfully.


In [13]:
callbacks_animal = [
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_cow_lumpy_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]
print("Callbacks ready.")

Callbacks ready.


In [14]:
EPOCHS = 15
history_animal = model_animal.fit(
    train_ds_animal,
    validation_data=val_ds_animal,
    epochs=EPOCHS,
    callbacks=callbacks_animal
)

Epoch 1/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5726 - loss: 0.6882
Epoch 1: val_accuracy improved from None to 0.75401, saving model to /kaggle/working/best_cow_lumpy_model.keras

Epoch 1: finished saving model to /kaggle/working/best_cow_lumpy_model.keras
24/24 ━━━━━━━━━━━━━━━━━━━━ 57s 2s/step - accuracy: 0.6622 - loss: 0.6065 - val_accuracy: 0.7540 - val_loss: 0.4935 - learning_rate: 0.0010
Epoch 2/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7879 - loss: 0.4306
Epoch 2: val_accuracy improved from 0.75401 to 0.80214, saving model to /kaggle/working/best_cow_lumpy_model.keras

Epoch 2: finished saving model to /kaggle/working/best_cow_lumpy_model.keras
24/24 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.7944 - loss: 0.4347 - val_accuracy: 0.8021 - val_loss: 0.4334 - learning_rate: 0.0010
Epoch 3/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8270 - loss: 0.3621
Epoch 3: val_accuracy did not improve from 0.80214
24/24 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step 

In [15]:
loss, accuracy = model_animal.evaluate(val_ds_animal)
print(f"Final Validation Accuracy: {accuracy*100:.2f}%")
print(f"Final Validation Loss: {loss:.4f}")

6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.8449 - loss: 0.3884
Final Validation Accuracy: 84.49%
Final Validation Loss: 0.3884


In [16]:
model_animal.save("/kaggle/working/cow_lumpy_final.keras")
print("Model saved!")

Model saved!


In [17]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_animal)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model_animal = converter.convert()

with open("/kaggle/working/cow_lumpy_model.tflite", "wb") as f:
    f.write(tflite_model_animal)

size_mb = os.path.getsize("/kaggle/working/cow_lumpy_model.tflite") / (1024*1024)
print(f"TFLite model size: {size_mb:.2f} MB")

INFO:tensorflow:Assets written to: /tmp/tmpyecft9zo/assets


INFO:tensorflow:Assets written to: /tmp/tmpyecft9zo/assets


Saved artifact at '/tmp/tmpyecft9zo'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_238')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  132355192350928: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  132355192351120: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  132356577792784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132356577802000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132356577801616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132356577802768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132356577800464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132356577800656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132356577803920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132356577803536: TensorSpec(shape=(), dtype=tf.resource, name=

W0000 00:00:1786888538.883727      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786888538.883772      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1786888539.149675      58 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


TFLite model size: 4.34 MB


In [18]:
import json

class_mapping_animal = {i: name for i, name in enumerate(class_names_animal)}

with open("/kaggle/working/cow_lumpy_class_mapping.json", "w") as f:
    json.dump(class_mapping_animal, f, indent=2)

print("Class mapping:", class_mapping_animal)

Class mapping: {0: 'healthycows', 1: 'lumpycows'}


In [19]:
import numpy as np

for images, labels in val_ds_animal.take(1):
    predictions = model_animal.predict(images)
    
    for i in range(5):
        true_class = class_names_animal[np.argmax(labels[i])]
        pred_class = class_names_animal[np.argmax(predictions[i])]
        confidence = np.max(predictions[i]) * 100
        print(f"True: {true_class} | Predicted: {pred_class} | Confidence: {confidence:.1f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
True: healthycows | Predicted: healthycows | Confidence: 51.0%
True: healthycows | Predicted: healthycows | Confidence: 98.4%
True: healthycows | Predicted: healthycows | Confidence: 99.3%
True: healthycows | Predicted: healthycows | Confidence: 87.8%
True: lumpycows | Predicted: lumpycows | Confidence: 99.0%


In [20]:
import shutil, os

EXPORT_DIR = "/kaggle/working/full_model_package"
os.makedirs(EXPORT_DIR, exist_ok=True)

files_to_copy = [
    # Plant model
    "/kaggle/working/plant_disease_model.tflite",
    "/kaggle/working/class_mapping.json",
    "/kaggle/working/plant_disease_final.keras",
    # Cow model
    "/kaggle/working/cow_lumpy_model.tflite",
    "/kaggle/working/cow_lumpy_class_mapping.json",
    "/kaggle/working/cow_lumpy_final.keras",
]

for f in files_to_copy:
    if os.path.exists(f):
        shutil.copy(f, EXPORT_DIR)
        print("Copied:", os.path.basename(f))
    else:
        print("Missing:", f)

shutil.make_archive("/kaggle/working/full_model_package", 'zip', EXPORT_DIR)
print("\n✅ Final zip ready: /kaggle/working/full_model_package.zip")

Missing: /kaggle/working/plant_disease_model.tflite
Missing: /kaggle/working/class_mapping.json
Missing: /kaggle/working/plant_disease_final.keras
Copied: cow_lumpy_model.tflite
Copied: cow_lumpy_class_mapping.json
Copied: cow_lumpy_final.keras

✅ Final zip ready: /kaggle/working/full_model_package.zip
